In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# 03 · Causal Analysis — DYNOTEARS on synthetic KPIs (CBMG)

Agent 3 (Causal Investigator) learns a **structural VAR** with DYNOTEARS (Pamfil et al., 2020):

$$X_t = X_t W + X_{t-1} A_1 + \dots + X_{t-p} A_p + \varepsilon_t$$

* `W` (d×d) — contemporaneous edges, constrained acyclic via the NOTEARS function $h(W)=\mathrm{tr}(e^{W\circ W})-d$
* `A` (p·d×d) — lagged edges; `A[k*d + i, j]` means metric *i* at *t−k−1* → metric *j* at *t*

Edges with |weight| ≥ 0.15 form the **Causal Business Metric Graph (CBMG)**. Because the synthetic data has a known
graph (`TRUE_EDGES`), we can score the recovered structure.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

from backend.demo.synthetic import generate, TRUE_EDGES
from backend.models.dynotears import dynotears, granger_matrix

## 1. Data → wide matrix
Pivot to one column per metric (index = timestamp).

In [ ]:
END = pd.Timestamp('2026-10-01 00:00', tz='UTC')
df = generate(periods=336, seed=42, end=END, incidents=True)
wide = df.pivot_table(index='timestamp', columns='metric_name', values='value').sort_index()
cols = list(wide.columns)
d = len(cols)
print(wide.shape)

## 2. Winsorise + standardise
Exactly as `CausalInvestigatorAgent._prepare`: clip each metric to median ± 3.5·MAD so the injected anomalies do not
dominate the structure (the graph should describe *normal* behaviour), then z-score. Standardising also removes the
scale/variance-ordering artefacts NOTEARS-type methods are sensitive to (Reisach et al., 2021).

In [ ]:
X = wide.to_numpy(dtype=float).copy()
med = np.median(X, axis=0)
mad = 1.4826 * np.median(np.abs(X - med), axis=0)
mad = np.where(mad < 1e-12, X.std(axis=0) + 1e-12, mad)
X = np.clip(X, med - 3.5 * mad, med + 3.5 * mad)
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-12)
print(X.shape, 'mean≈0:', np.allclose(X.mean(0), 0), 'std≈1:', np.allclose(X.std(0), 1))

## 3. Run DYNOTEARS (p = 1 lag)

In [ ]:
import time
t0 = time.perf_counter()
res = dynotears(X, p=1, lambda_w=0.05, lambda_a=0.05)
print(f'done in {time.perf_counter() - t0:.1f}s; acyclicity h(W) = {res.h:.2e}; W {res.W.shape}, A {res.A.shape}')

## 4. Threshold and extract edges
Self-loops in `A` are autoregression (allowed by the model) and are not causal edges between metrics, so they are excluded.

In [ ]:
THRESHOLD = 0.15

def extract_edges(W, A, cols, thr=THRESHOLD):
    d = len(cols)
    rows = []
    for i in range(d):
        for j in range(d):
            if i == j:
                continue
            if abs(W[i, j]) >= thr:
                rows.append((cols[i], cols[j], W[i, j], 0))
            for k in range(A.shape[0] // d):
                w = A[k * d + i, j]
                if abs(w) >= thr:
                    rows.append((cols[i], cols[j], w, k + 1))
    return pd.DataFrame(rows, columns=['cause', 'effect', 'weight', 'lag'])

learned = extract_edges(res.W, res.A, cols)
learned.sort_values('weight', key=abs, ascending=False).round(3)

## 5. Compare with the ground truth
An edge counts as correct when the directed pair (cause → effect) is in `TRUE_EDGES` (we also report whether the lag
and sign match). **Structural Hamming distance (SHD)** = missing edges + extra edges + reversed edges, where a reversed
edge counts once.

In [ ]:
true_set = {(c, e) for c, e, _, _ in TRUE_EDGES}
pred_set = set(zip(learned['cause'], learned['effect']))

tp = pred_set & true_set
fp = pred_set - true_set
fn = true_set - pred_set
precision = len(tp) / len(pred_set) if pred_set else 0.0
recall    = len(tp) / len(true_set)
f1        = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

reversed_ = {(c, e) for (c, e) in fp if (e, c) in fn}
shd = len(fn - {(e, c) for (c, e) in reversed_}) + len(fp - reversed_) + len(reversed_)

print(f'true edges: {len(true_set)} | learned edges: {len(pred_set)}')
print(f'TP={len(tp)}  FP={len(fp)}  FN={len(fn)}  reversed={len(reversed_)}')
print(f'precision = {precision:.3f}')
print(f'recall    = {recall:.3f}')
print(f'F1        = {f1:.3f}')
print(f'SHD       = {shd}')

In [ ]:
truth = pd.DataFrame(TRUE_EDGES, columns=['cause', 'effect', 'true_coef', 'true_lag'])
cmp = truth.merge(learned, on=['cause', 'effect'], how='outer', indicator=True)
cmp['status'] = cmp['_merge'].map({'both': 'TP', 'left_only': 'FN (missed)', 'right_only': 'FP (extra)'})
cmp['sign_ok'] = np.where(cmp['status'] == 'TP', np.sign(cmp['true_coef']) == np.sign(cmp['weight']), np.nan)
cmp['lag_ok'] = np.where(cmp['status'] == 'TP', cmp['true_lag'] == cmp['lag'], np.nan)
cmp.drop(columns='_merge').round(3)

## 6. Baseline: lagged-regression (Granger-style) matrix
`granger_matrix` is Agent 3's last-resort fallback. Same threshold, for comparison.

In [ ]:
G = granger_matrix(X, p=1)
np.fill_diagonal(G, 0)
g_pred = {(cols[i], cols[j]) for i in range(d) for j in range(d) if G[i, j] >= THRESHOLD}
g_tp = len(g_pred & true_set)
g_p = g_tp / len(g_pred) if g_pred else 0.0
g_r = g_tp / len(true_set)
g_f1 = 2 * g_p * g_r / (g_p + g_r) if g_p + g_r else 0.0
pd.DataFrame({'precision': [precision, g_p], 'recall': [recall, g_r], 'F1': [f1, g_f1],
              'edges': [len(pred_set), len(g_pred)]}, index=['DYNOTEARS', 'Granger (lagged OLS)']).round(3)

## 7. Draw the learned graph
Green = correct edge, red = extra edge, grey dashed = missed true edge. Edge labels show weight and lag.

In [ ]:
Gx = nx.DiGraph()
Gx.add_nodes_from(cols)
for _, r in learned.iterrows():
    Gx.add_edge(r['cause'], r['effect'], weight=r['weight'], lag=r['lag'])

pos = nx.spring_layout(Gx, seed=7, k=1.2)
fig, ax = plt.subplots(figsize=(13, 9))
nx.draw_networkx_nodes(Gx, pos, node_color='#1E7A78', node_size=1600, ax=ax)
nx.draw_networkx_labels(Gx, pos, font_size=7, font_color='white', ax=ax)
good = [(u, v) for u, v in Gx.edges if (u, v) in true_set]
bad  = [(u, v) for u, v in Gx.edges if (u, v) not in true_set]
nx.draw_networkx_edges(Gx, pos, edgelist=good, edge_color='tab:green', width=2, arrowsize=18,
                       node_size=1600, ax=ax)
nx.draw_networkx_edges(Gx, pos, edgelist=bad, edge_color='tab:red', width=2, arrowsize=18, node_size=1600, ax=ax)
nx.draw_networkx_edges(Gx, pos, edgelist=list(fn), edge_color='0.6', style='dashed', width=1.2, arrowsize=14,
                       node_size=1600, ax=ax)
labels = {(u, v): f"{dd['weight']:.2f} (lag {dd['lag']})" for u, v, dd in Gx.edges(data=True)}
nx.draw_networkx_edge_labels(Gx, pos, edge_labels=labels, font_size=7, ax=ax)
ax.set_title(f'Learned CBMG — precision {precision:.2f}, recall {recall:.2f}, SHD {shd}')
ax.axis('off')
plt.tight_layout()
plt.show()

## 8. Root-cause walk (illustration)
Agent 3 walks the graph backwards from an anomalous metric. Here: all ancestors of `daily_revenue` and the strongest
path from `ad_spend_usd`, using the project's `CausalBusinessMetricGraph`.

In [ ]:
from backend.models.cbmg import CausalBusinessMetricGraph

cbmg = CausalBusinessMetricGraph()
for c in cols:
    cbmg.add_node(c)
for _, r in learned.iterrows():
    cbmg.add_edge(r['cause'], r['effect'], weight=abs(r['weight']), lag=int(r['lag']), sign=int(np.sign(r['weight'])))

print('ancestors of daily_revenue:', sorted(cbmg.get_ancestors('daily_revenue')))
path, strength, lag = cbmg.strongest_path('ad_spend_usd', 'daily_revenue')
print('strongest path:', ' -> '.join(path) if path else '(none)', f'| strength {strength:.3f} | total lag {lag} h')

## Caveats
The synthetic generator is a *linear* SVAR with Gaussian noise and lag-1 edges — exactly DYNOTEARS' assumptions — and
all relevant variables are observed (causal sufficiency). These scores are therefore a sanity check / upper bound, not
evidence of accuracy on real business data. See `research/gap_analysis.md` → *Limitations & future work*.